# Churn Batch Scoring

Loads the registered RandomForest model (`workspace.default.churn_rf_model` v1) from Unity Catalog, scores all rows in the silver table, and writes predictions to `workspace.default.churn_predictions`. Designed to run as a Lakeflow Job after the ETL pipeline updates the silver table.

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = "workspace.default.churn_rf_model"
MODEL_VERSION = "1"

# Load as sklearn flavor to preserve predict_proba and classes_
model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{MODEL_VERSION}")
print(f"✓ Loaded model: {MODEL_NAME} version {MODEL_VERSION}")

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# --- Read silver table ---
SILVER_TABLE = "workspace.default.telco_customer_churn_silver"
PREDICTIONS_TABLE = "workspace.default.churn_predictions"

silver_df = spark.table(SILVER_TABLE)
pdf = silver_df.toPandas()
print(f"Read {len(pdf)} rows from {SILVER_TABLE}")

# --- Prepare features (same as training: drop customerID and Churn) ---
customer_ids = pdf["customerID"].copy()
features = pdf.drop(columns=["customerID", "Churn"])

# Convert TotalCharges from string to numeric (same as training preprocessing)
# Some values are " " (spaces) which become NaN, handled by the pipeline's imputer
features["TotalCharges"] = pd.to_numeric(features["TotalCharges"], errors="coerce")

# --- Predict ---
predictions = model.predict(features)

# Get churn probability (class "Yes") if available
try:
    proba = model.predict_proba(features)
    # Find the index of "Yes" in the model's classes_
    classes = model.classes_ if hasattr(model, "classes_") else None
    if classes is not None and "Yes" in classes:
        yes_idx = list(classes).index("Yes")
        churn_proba = proba[:, yes_idx]
    else:
        churn_proba = proba[:, -1]  # fallback: last column
except Exception:
    churn_proba = [None] * len(predictions)

# --- Build output DataFrame ---
output = pd.DataFrame({
    "customerID": customer_ids.values,
    "churn_prediction": predictions,
    "churn_probability": churn_proba,
    "prediction_timestamp": pd.Timestamp.now(),
})

print(f"Predictions: {len(output)} rows")
print(output.head(10))

# --- Write to Delta table ---
spark.createDataFrame(output).write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(PREDICTIONS_TABLE)

print(f"✓ Written to {PREDICTIONS_TABLE}")

# --- Display predictions ---
display(spark.table(PREDICTIONS_TABLE))

# --- Summary ---
summary = spark.table(PREDICTIONS_TABLE).groupBy("churn_prediction").count()
print("--- Prediction Summary ---")
display(summary)